# Kiểm thử Phiên bản Cũ: Mô hình 3 Khối Fusion & Prompt Ban Đầu
Notebook này được thiết kế đặc biệt để tải trọng số `archive/cross_attention_3blocks/best_model.pth`.
Các lớp `Conv1x1Classifier` và `CrossAttention` đã được tái tạo lại đúng cấu trúc nguyên bản để tương thích 100% với file weights cũ mà không bị lỗi.

In [1]:
import os
import sys
import time
from pathlib import Path
import torch
import torch.nn as nn
from PIL import Image
from torchvision import transforms
from transformers import AutoTokenizer, AutoModel, AutoProcessor, LlavaForConditionalGeneration, BitsAndBytesConfig
from tqdm.notebook import tqdm
from sklearn.metrics import accuracy_score

current_dir = Path.cwd()
PROJECT_ROOT = current_dir
while not (PROJECT_ROOT / 'src').exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT))
print(f"Project Root: {PROJECT_ROOT}")

from src.datasets.multimodal_raw_dataset import MultiModalRawDataset
from src.models.backbones.vision.convnext_cbam import ConvNeXt_CBAM
from src.models.fusion.mulco_fusion import RestormerBlock
from src.models.backbones.text.clip_text_encoder import CLIPTextEncoder

Project Root: /media/data3/users/luongdth/MulCo-PlantNet


## 1. Tái tạo lại Kiến trúc Cũ (Tương thích với best_model.pth)

In [2]:
# 1. Classifier cũ sử dụng Flatten và Fully Connected (chưa có GAP)
class Conv1x1ClassifierOld(nn.Module):
    def __init__(self, in_channels, num_classes, spatial_size=(7, 7)):
        super().__init__()
        self.conv_1x1 = nn.Conv2d(in_channels, num_classes, kernel_size=1)
        self.act = nn.GELU()
        self.dropout = nn.Dropout(p=0.4)
        flatten_dim = num_classes * spatial_size[0] * spatial_size[1]
        self.fc = nn.Linear(flatten_dim, num_classes)

    def forward(self, x):
        x = self.conv_1x1(x)
        x = self.act(x)
        x = x.flatten(start_dim=1)
        x = self.dropout(x)
        x = self.fc(x)
        return x

# 2. CrossAttention cũ (Dùng Linear cho Text-to-Value thay vì Conv2d)
class CrossAttentionOld(nn.Module):
    def __init__(self, dim, num_heads=8):
        super().__init__()
        self.num_heads = num_heads
        self.scale = (dim // num_heads) ** -0.5
        self.q_proj = nn.Conv2d(dim, dim, kernel_size=1)
        self.k_proj = nn.Linear(dim, dim)
        self.v_proj = nn.Linear(dim, dim)
        self.out_proj = nn.Conv2d(dim, dim, kernel_size=1)

    def forward(self, img_feat, txt_feat):
        b, c, h, w = img_feat.shape
        _, l, _ = txt_feat.shape
        q = self.q_proj(img_feat).view(b, self.num_heads, c // self.num_heads, h * w).transpose(-2, -1)
        k = self.k_proj(txt_feat).view(b, l, self.num_heads, c // self.num_heads).transpose(1, 2)
        v = self.v_proj(txt_feat).view(b, l, self.num_heads, c // self.num_heads).transpose(1, 2)
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        out = (attn @ v).transpose(-2, -1).reshape(b, c, h, w)
        return self.out_proj(out)

# 3. Khối Fusion 3 Blocks
class MulCoFusionBlockOld(nn.Module):
    def __init__(self, dim, num_heads=8):
        super().__init__()
        self.cross_attn = CrossAttentionOld(dim, num_heads)
        self.restormer = RestormerBlock(dim, num_heads)

    def forward(self, img_feat, txt_feat):
        guided_img = img_feat + self.cross_attn(img_feat, txt_feat)
        refined_img = self.restormer(guided_img)
        return refined_img, txt_feat

class MulCoEndToEnd3Blocks(nn.Module):
    def __init__(self, num_classes=28, proj_dim=512, device="cpu"):
        super().__init__()
        self.image_backbone = ConvNeXt_CBAM(num_classes=num_classes)
        self.text_backbone = CLIPTextEncoder(model_name="ViT-B-32", pretrained="openai", device=device)
        
        self.img_proj = nn.Conv2d(1024, proj_dim, kernel_size=1)
        self.txt_proj = nn.Linear(512, proj_dim)
        
        self.fusion_blocks = nn.ModuleList([
            MulCoFusionBlockOld(dim=proj_dim, num_heads=8) for _ in range(3)
        ])
        
        self.classifier = Conv1x1ClassifierOld(in_channels=proj_dim, num_classes=num_classes, spatial_size=(7, 7))

    def forward(self, images, captions):
        img_feat = self.image_backbone.forward_features_spatial(images) 
        txt_feat = self.text_backbone(captions)
        txt_feat = txt_feat.unsqueeze(1)
        
        img_feat = self.img_proj(img_feat)
        txt_feat = self.txt_proj(txt_feat)
        
        for block in self.fusion_blocks:
            img_feat, txt_feat = block(img_feat, txt_feat)
            
        return self.classifier(img_feat)

## 2. Khởi tạo Models và Load Trọng số cũ

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

print("Loading LLaVA-1.5-7B...")
llava_processor = AutoProcessor.from_pretrained("llava-hf/llava-1.5-7b-hf")
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)
llava_model = LlavaForConditionalGeneration.from_pretrained(
    "llava-hf/llava-1.5-7b-hf",
    quantization_config=quantization_config,
    device_map="auto"
)

print("Loading Old 3-Block MulCo...")
mulco_model = MulCoEndToEnd3Blocks(num_classes=28, device=str(device)).to(device)
ckpt_path = os.path.join(PROJECT_ROOT, "archive", "cross_attention_3blocks", "best_model.pth")

# Sử dụng strict=False để bỏ qua các keys của BatchNorm (nếu mô hình cực cũ chưa có BN ở head)
mulco_model.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=True), strict=False)
mulco_model.eval()

mulco_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

Using device: cuda
Loading LLaVA-1.5-7B...


Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

Loading Old 3-Block MulCo...


open_clip_model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

/media/data3/users/luongdth/anaconda3/envs/gr1/lib/python3.12/site-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


## 3. Hàm Dự đoán sử dụng Prompt Ban Đầu

In [4]:
def predict_end_to_end(image_path):
    raw_image = Image.open(image_path).convert("RGB")
    
    # Sinh Text từ LLaVA với prompt CŨ
    prompt_text = (
        "Please describe the image of the plant leaf according to the following guidelines:\n"
        "Step1, Identify the color of the leaf, including the base color and any phenotypic characteristics of spots or discolored areas, such as their location, size, length, number, and color.\n"
        "Step2, describe the overall shape of the leaf, including whether it is round, oval, heart-shaped, or another shape.\n"
        "Step3, describe the texture of the leaf, including whether the surface is smooth, hairy, or has other features.\n"
        "Step4, if the leaf has edges, describe the characteristics of the edges, such as whether they are smooth, serrated, or wavy.\n"
        "Step5, describe whether there are any visible damages on the leaf, such as holes, tears, wilting, or lesions.\n"
        "Step6, if there are veins on the leaf, describe their distribution and color.\n"
        "Step7, if the image includes the petiole of the leaf, describe the thickness, color, and texture of the petiole.\n"
        "Please keep the description objective, providing only the information visible in the image without adding any inferences or explanations.\n"
    )
    prompt = f"USER: <image>\n{prompt_text}\nASSISTANT:"
    inputs = llava_processor(text=prompt, images=raw_image, return_tensors="pt").to(device, torch.float16)
    
    with torch.no_grad():
        output = llava_model.generate(
            **inputs, 
            max_new_tokens=256,
            num_beams=3,
            do_sample=False
        )
    generated_text = llava_processor.decode(output[0], skip_special_tokens=True)
    caption = generated_text.split("ASSISTANT:")[-1].strip()
    
    # Tiền xử lý cho MulCo
    img_tensor = mulco_transform(raw_image).unsqueeze(0).to(device)
    
    # Đưa vào MulCo dự đoán
    with torch.no_grad():
        logits = mulco_model(img_tensor, [caption])
        pred = torch.argmax(logits, dim=1).item()
        
    torch.cuda.empty_cache()
    return pred, caption

## 4. Vòng lặp Đánh giá

In [5]:
class_mapping = {'Apple_Scab_Leaf': 0, 'Apple_leaf': 1, 'Apple_rust_leaf': 2, 'Bell_pepper_leaf': 3, 'Bell_pepper_leaf_spot': 4, 'Blueberry_leaf': 5, 'Cherry_leaf': 6, 'Corn_Gray_leaf_spot': 7, 'Corn_leaf_blight': 8, 'Corn_rust_leaf': 9, 'Peach_leaf': 10, 'Potato_leaf_early_blight': 11, 'Potato_leaf_late_blight': 12, 'Raspberry_leaf': 13, 'Soyabean_leaf': 14, 'Squash_Powdery_mildew_leaf': 15, 'Strawberry_leaf': 16, 'Tomato_Early_blight_leaf': 17, 'Tomato_Septoria_leaf_spot': 18, 'Tomato_leaf': 19, 'Tomato_leaf_bacterial_spot': 20, 'Tomato_leaf_late_blight': 21, 'Tomato_leaf_mosaic_virus': 22, 'Tomato_leaf_yellow_virus': 23, 'Tomato_mold_leaf': 24, 'Tomato_two_spotted_spider_mites_leaf': 25, 'grape_leaf': 26, 'grape_leaf_black_rot': 27}
idx_to_class = {v: k for k, v in class_mapping.items()}

test_dataset = MultiModalRawDataset(
    image_root=os.path.join(PROJECT_ROOT, "data/processed/PlantDocSplited_depth_AUG/test"),
    caption_root=os.path.join(PROJECT_ROOT, "data/processed/captions_LLaVA_depth_AUG/test"),
    transform=None,  # Để None vì ảnh sẽ được xử lý trong hàm predict_end_to_end
    use_depth_suppressed=False,
    strict_caption_match=False
)
print(f"Total items to test: {len(test_dataset)}")

all_preds = []
all_labels = []
num_samples_to_test = len(test_dataset)

for idx in tqdm(range(num_samples_to_test), desc="End-to-End Inference (Old 3-Blocks)"):
    item = test_dataset[idx]
    image_path = item["image_path"]
    true_label = item["label"]
    
    start_time = time.time()
    pred, generated_caption = predict_end_to_end(image_path)
    latency = time.time() - start_time
    
    print(f"\n--- Image: {Path(image_path).name} | Time: {latency:.2f}s ---")
    print(f"Generated Caption:\n{generated_caption}")
    print(f"-> Predicted class: {idx_to_class[pred]} ({pred}) | True class: {idx_to_class[true_label]} ({true_label})")
    
    all_preds.append(pred)
    all_labels.append(true_label)

acc = accuracy_score(all_labels, all_preds)
print(f"\n====================================")
print(f"End-to-End Accuracy (Old Model): {acc:.4f}")
print(f"====================================")


[MultiModalRawDataset] Total selected images: 250
[MultiModalRawDataset] Valid samples: 250
[MultiModalRawDataset] Skipped missing caption: 0
[MultiModalRawDataset] Skipped invalid caption: 0
[MultiModalRawDataset] Matched by external mapping: 0
[MultiModalRawDataset] Num classes: 28
[MultiModalRawDataset] class_to_idx: {'Apple_Scab_Leaf': 0, 'Apple_leaf': 1, 'Apple_rust_leaf': 2, 'Bell_pepper_leaf': 3, 'Bell_pepper_leaf_spot': 4, 'Blueberry_leaf': 5, 'Cherry_leaf': 6, 'Corn_Gray_leaf_spot': 7, 'Corn_leaf_blight': 8, 'Corn_rust_leaf': 9, 'Peach_leaf': 10, 'Potato_leaf_early_blight': 11, 'Potato_leaf_late_blight': 12, 'Raspberry_leaf': 13, 'Soyabean_leaf': 14, 'Squash_Powdery_mildew_leaf': 15, 'Strawberry_leaf': 16, 'Tomato_Early_blight_leaf': 17, 'Tomato_Septoria_leaf_spot': 18, 'Tomato_leaf': 19, 'Tomato_leaf_bacterial_spot': 20, 'Tomato_leaf_late_blight': 21, 'Tomato_leaf_mosaic_virus': 22, 'Tomato_leaf_yellow_virus': 23, 'Tomato_mold_leaf': 24, 'Tomato_two_spotted_spider_mites_leaf'

End-to-End Inference (Old 3-Blocks):   0%|          | 0/250 [00:00<?, ?it/s]


--- Image: test_Apple Scab Leaf_1.jpg | Time: 17.51s ---
Generated Caption:
Step 1: The leaf is green with brown spots. The base color is green, and the spots are brown.
Step 2: The leaf is heart-shaped.
Step 3: The surface of the leaf is smooth.
Step 4: The edges of the leaf are smooth.
Step 5: There are no visible damages on the leaf.
Step 6: The veins on the leaf are green.
Step 7: The petiole of the leaf is thin and green.
-> Predicted class: Apple_rust_leaf (2) | True class: Apple_Scab_Leaf (0)

--- Image: test_Apple Scab Leaf_10.jpg | Time: 7.83s ---
Generated Caption:
The leaf is green with brown spots, and it has a heart-shaped shape. The surface of the leaf is smooth, and the edges are serrated. There are no visible damages on the leaf, and the veins are green. The petiole of the leaf is thin and green.
-> Predicted class: Apple_rust_leaf (2) | True class: Apple_Scab_Leaf (0)

--- Image: test_Apple Scab Leaf_2.jpg | Time: 11.49s ---
Generated Caption:
Step 1: The leaf is gree

KeyboardInterrupt: 